In [41]:
import pandas as pd
from sklearn.model_selection import train_test_split


train_ds = pd.read_csv('train_runes.csv')
train_ds, validation_ds = train_test_split(train_ds, test_size=0.2, random_state=42)
test_ds = pd.read_csv('test_runes.csv')

print('intersection: {}'.format(
    len(set(train_ds['rune'].values) & set(test_ds['rune'].values))
))

intersection: 0


In [42]:
import numpy as np

RUNE_ENCODING = {'F': [1, 0, 0], 'W': [0, 1, 0], 'E': [0, 0, 1]}

def encode_sequence(seq):
    return [bit for letter in seq for bit in RUNE_ENCODING[letter]]

X_train = np.array(train_ds['rune'].apply(encode_sequence).tolist())
X_val = np.array(validation_ds['rune'].apply(encode_sequence).tolist())
X_test = np.array(test_ds['rune'].apply(encode_sequence).tolist())

y_train = train_ds['spell'].values
y_val = validation_ds['spell'].values

print(X_train.shape)  # (170, 15)
print(X_train[0])     # e.g. [0 1 0  0 1 0  0 1 0  0 1 0  1 0 0] for WWWWF

(136, 15)
[0 0 1 0 0 1 0 1 0 0 1 0 0 1 0]


In [46]:
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier

k_best, acc_best = 0, 0
for k in range(1, 10):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    val_preds = knn.predict(X_val)

    acc = accuracy_score(y_val, val_preds)
    if acc > acc_best:
        acc_best = acc
        k_best = k

    print('accuracy (k={}): '.format(k), acc)

knn = KNeighborsClassifier(n_neighbors=k_best)
knn.fit(X_train, y_train)

test_preds = knn.predict(X_test)
result = test_ds.copy()
result['spell'] = test_preds.astype(int)
result.to_csv('answers.csv', index=False)

accuracy (k=1):  0.8529411764705882
accuracy (k=2):  0.7941176470588235
accuracy (k=3):  0.9411764705882353
accuracy (k=4):  0.8823529411764706
accuracy (k=5):  0.9705882352941176
accuracy (k=6):  0.8823529411764706
accuracy (k=7):  0.9117647058823529
accuracy (k=8):  0.8823529411764706
accuracy (k=9):  0.9117647058823529


In [48]:
from sklearn.naive_bayes import MultinomialNB

classifier = MultinomialNB()
classifier.fit(X_train, y_train)
val_preds = classifier.predict(X_val)
print('accuracy: ', accuracy_score(y_val, val_preds))

test_preds = classifier.predict(X_test)
result = test_ds.copy()
result['spell'] = test_preds.astype(int)
result.to_csv('answers.csv', index=False)

accuracy:  0.9117647058823529


In [58]:
from sklearn.ensemble import RandomForestClassifier

classifier = RandomForestClassifier(n_estimators=150, random_state=42)
classifier.fit(X_train, y_train)
val_preds = classifier.predict(X_val)
print('accuracy: ', accuracy_score(y_val, val_preds))

test_preds = classifier.predict(X_test)
result = test_ds.copy()
result['spell'] = test_preds.astype(int)
result.to_csv('answers.csv', index=False)

accuracy:  1.0
